In [12]:
// A global variable to hold the reference (works in Jupyter/TSLab)
if (typeof (global as any).mainSub !== "undefined") {
  (global as any).mainSub.unsubscribe();
  console.log("Cleaned up old subscription.");
}


Cleaned up old subscription.


In [13]:
const ranks = ['6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A'];
const suits = ['hearts', 'diamonds', 'clubs', 'spades'];

const rankValues: Record<string, number> = { 
    '6': 6, '7': 7, '8': 8, '9': 9, '10': 10, 'J': 11, 'Q': 12, 'K': 13, 'A': 14 
};

interface Card {
    id: string;
    rank: typeof ranks[number];
    suit: typeof suits[number];
}

interface Player {
    id: number;
    name: string;
    hand: Card[];
    isDealer: boolean;
}
    

In [14]:
import { shuffle, uniqueId } from 'underscore'

function getDeck() {
  return Object.entries(rankValues).flatMap(([rank, value]) => suits.map(suit => ({rank, value, suit, id: uniqueId()})))
}

const deck = shuffle(getDeck())
console.table(deck)


┌─────────┬──────┬───────┬────────────┬──────┐
│ (index) │ rank │ value │ suit       │ id   │
├─────────┼──────┼───────┼────────────┼──────┤
│ 0       │ 'Q'  │ 12    │ 'diamonds' │ '62' │
│ 1       │ 'A'  │ 14    │ 'clubs'    │ '71' │
│ 2       │ 'K'  │ 13    │ 'spades'   │ '68' │
│ 3       │ '7'  │ 7     │ 'clubs'    │ '43' │
│ 4       │ '10' │ 10    │ 'spades'   │ '56' │
│ 5       │ '6'  │ 6     │ 'clubs'    │ '39' │
│ 6       │ '8'  │ 8     │ 'clubs'    │ '47' │
│ 7       │ '9'  │ 9     │ 'spades'   │ '52' │
│ 8       │ '7'  │ 7     │ 'hearts'   │ '41' │
│ 9       │ '8'  │ 8     │ 'hearts'   │ '45' │
│ 10      │ 'K'  │ 13    │ 'clubs'    │ '67' │
│ 11      │ 'J'  │ 11    │ 'spades'   │ '60' │
│ 12      │ 'Q'  │ 12    │ 'hearts'   │ '61' │
│ 13      │ '10' │ 10    │ 'diamonds' │ '54' │
│ 14      │ 'Q'  │ 12    │ 'clubs'    │ '63' │
│ 15      │ '7'  │ 7     │ 'spades'   │ '44' │
│ 16      │ '9'  │ 9     │ 'hearts'   │ '49' │
│ 17      │ '9'  │ 9     │ 'diamonds' │ '50' │
│ 18      │ '

In [15]:
import { BehaviorSubject } from 'rxjs'

const table$ = new BehaviorSubject({
  currentTurnId: 0,
  currentDefendId: 1,
  deck: deck as Card[],
  trumps: deck[0],
  players: [
    { id: 0, name: "Player 1" },
    { id: 1, name: "Player 2" },
    { id: 2, name: "Player 3" },
    { id: 3, name: "Player 4" },
  ],
  hands: [
    {
      playerId: 0,
      cards: [] as Card[]
    },
    {
      playerId: 1,
      cards: [] as Card[]
    },
    {
      playerId: 2,
      cards: [] as Card[]
    },
    {
      playerId: 3,
      cards: [] as Card[]
    }
  ],
  attack: [] as Card[],
  beaten: [] as Card[],
});

In [16]:
import { distinctUntilChanged, map } from 'rxjs'

const currentPlayer$ = table$.pipe(
    map(table => table.players.find(p => p.id === table.currentTurnId)),
    distinctUntilChanged()
)

currentPlayer$.subscribe((player) => {
  console.log(`It is now player ${player?.name}'s turn!`);
});
    

It is now player Player 1's turn!
<ref *1> SafeSubscriber {
  initialTeardown: undefined,
  closed: false,
  _parentage: null,
  _finalizers: [
    OperatorSubscriber {
      initialTeardown: undefined,
      closed: false,
      _parentage: [Circular *1],
      _finalizers: [Array],
      isStopped: false,
      destination: [Circular *1],
      onFinalize: undefined,
      shouldUnsubscribe: undefined,
      _next: [Function (anonymous)],
      _error: [Function (anonymous)],
      _complete: [Function (anonymous)]
    }
  ],
  isStopped: false,
  destination: ConsumerObserver {
    partialObserver: {
      next: [Function (anonymous)],
      error: undefined,
      complete: undefined
    }
  }
}


In [17]:
const dealCards = () => {
  const currentTable = table$.value;
  let newHands = { ...currentTable.hands };
  let newDeck = [...currentTable.deck]; // Copy the deck
  currentTable.players.forEach((player) => {
    const cardsToTake = 6 - (newHands[player.id]?.cards.length || 0);
    if (cardsToTake > 0 && newDeck.length > 0) {
      const taken = newDeck.splice(0, cardsToTake);
      newHands[player.id].cards.push(...taken);
      console.log(`${player.name} draws ${taken.length} cards. (${newDeck.length} left in deck)`);
    }
  });
  table$.next({ ...currentTable, hands: newHands, deck: newDeck });
};


In [18]:
const shiftNextPlayerId = (id: number) => {
  const currentTable = table$.value;
  const index = currentTable.players.findIndex((p) => p.id === id);
  return currentTable.players[(index + 1) % currentTable.players.length].id;
};


In [19]:
const attack = () => {
  const currentTable = table$.value;
  const currentPlayer = currentTable.players.find(
    (p) => p.id === currentTable.currentTurnId,
  );
  if (!currentPlayer) return;

  if (currentTable.hands[currentPlayer.id].cards.length === 0) {
    console.log(`${currentPlayer.name} is empty! Skipping turn...`);
    table$.next({
      ...currentTable,
      currentTurnId: shiftNextPlayerId(currentTable.currentTurnId),
      currentDefendId: shiftNextPlayerId(
        shiftNextPlayerId(currentTable.currentTurnId),
      ),
    });
    return;
  }

  const card = currentTable.hands[currentPlayer.id].cards.pop();
  if (!card) return;

  currentTable.attack.push(card);
  console.log(
    `${currentPlayer.name} attacks with ${card.rank} of ${card.suit}`,
  );

  table$.next({ ...currentTable });
};


In [20]:
const defend = () => {
  const currentTable = table$.value;
  const defender = currentTable.players.find(
    (p) => p.id === currentTable.currentDefendId,
  );

  const attackCard = currentTable.attack[currentTable.attack.length - 1];
  const defenderHand = currentTable.hands[currentTable.currentDefendId];

  const defendCard = defenderHand?.cards.find(
    (c) =>
      rankValues[c.rank] > rankValues[attackCard.rank] &&
      (c.suit === attackCard.suit || c.suit === currentTable.trumps.suit),
  );

  const passCard = defenderHand?.cards.find(
    (c) => rankValues[c.rank] === rankValues[attackCard.rank]
  );

  if (passCard) {
    defenderHand?.cards.splice(defenderHand.cards.indexOf(passCard), 1);
    currentTable.attack.push(passCard);
    currentTable.currentTurnId = defender.id;
    currentTable.currentDefendId = shiftNextPlayerId(currentTable.currentTurnId);

    console.log(
      `player ${defender?.name} passed with ${passCard?.rank} of ${passCard?.suit}`,
    );
  } else if (defendCard) {
    defenderHand?.cards.splice(defenderHand.cards.indexOf(defendCard), 1);
    currentTable.beaten.push(attackCard);
    currentTable.beaten.push(defendCard);
    currentTable.attack = [];
     currentTable.currentTurnId = defender.id;
     currentTable.currentDefendId = shiftNextPlayerId(
       currentTable.currentTurnId,
     );


    console.log(
      `player ${defender?.name} defended with ${defendCard?.rank} of ${defendCard?.suit}`,
    );
  } else {
    console.log(
      `player ${defender?.name} could not defend ${attackCard?.rank} of ${attackCard?.suit}`,
    );

    defenderHand?.cards.push(attackCard);
    currentTable.attack = [];
    currentTable.currentTurnId = shiftNextPlayerId(defender.id);
    currentTable.currentDefendId = shiftNextPlayerId(currentTable.currentTurnId);

  }

  if (currentTable.currentTurnId === currentTable.currentDefendId) {
    console.error("SYNC ERROR: Attacker and Defender are the same player!");
  }

  table$.next(currentTable);
};


In [21]:
const isGameOver = () => {
  const hands = Object.values(table$.value.hands);
  const deck = table$.value.deck;
  const players = table$.value.players;

  // 1. If the deck still has cards, the game is definitely NOT over
  if (deck.length > 0) {
    return false;
  }

  // 2. Check if ANY player actually has an empty hand
  const winningHand = hands.find((h) => h.cards.length === 0);

  // 3. If no one has an empty hand yet (even with empty deck), the game continues
  if (!winningHand) {
    return false;
  }

  // 4. Now it's safe to read the properties!
  const winnerId = winningHand.playerId;
  const winnerName = players.find((p) => p.id === winnerId)?.name;

  console.log(`--- Game Over! Winner is ${winnerName} ---`);

  // Clean up
  if ((global as any).mainSub) {
    (global as any).mainSub.unsubscribe();
  }
  table$.complete();
  return true;
};


In [ ]:
import { asyncScheduler, observeOn, tap } from "rxjs";

(global as any).mainSub = table$
  .pipe(
    observeOn(asyncScheduler),
    map((t) => t.currentTurnId),
    distinctUntilChanged(),
    tap(() => {
      // Phase 1: Draw cards (updates state via .next)
      dealCards();

      // Phase 2: Attempt Attack (updates state via .next)
      attack();

      // Phase 3: Attept Defense (updates state via .next)
      defend();

      // Phase 4: Check if finished
      isGameOver();
    }),
  )
  .subscribe();


<ref *1> SafeSubscriber {
  initialTeardown: undefined,
  closed: false,
  _parentage: null,
  _finalizers: [
    OperatorSubscriber {
      initialTeardown: undefined,
      closed: false,
      _parentage: [Circular *1],
      _finalizers: [Array],
      isStopped: false,
      destination: [Circular *1],
      onFinalize: [Function (anonymous)],
      shouldUnsubscribe: undefined,
      _next: [Function (anonymous)],
      _error: [Function (anonymous)],
      _complete: [Function (anonymous)]
    }
  ],
  isStopped: false,
  destination: ConsumerObserver {
    partialObserver: { next: undefined, error: undefined, complete: undefined }
  }
}
Player 1 draws 6 cards. (30 left in deck)
Player 2 draws 6 cards. (24 left in deck)
Player 3 draws 6 cards. (18 left in deck)
Player 4 draws 6 cards. (12 left in deck)
Player 1 attacks with 6 of clubs
player Player 2 defended with 8 of clubs
It is now player Player 2's turn!


Player 1 draws 1 cards. (11 left in deck)
Player 2 draws 1 cards. (10 left in deck)
Player 2 attacks with 9 of clubs
player Player 3 passed with 9 of hearts
It is now player Player 3's turn!
Player 2 draws 1 cards. (9 left in deck)
Player 3 draws 1 cards. (8 left in deck)
Player 3 attacks with K of diamonds
player Player 4 defended with A of diamonds
It is now player Player 4's turn!
Player 3 draws 1 cards. (7 left in deck)
Player 4 draws 1 cards. (6 left in deck)
Player 4 attacks with K of hearts
player Player 1 passed with K of spades
It is now player Player 1's turn!
Player 1 draws 1 cards. (5 left in deck)
Player 4 draws 1 cards. (4 left in deck)
Player 1 attacks with A of spades
player Player 2 could not defend A of spades
It is now player Player 3's turn!
Player 1 draws 1 cards. (3 left in deck)
Player 3 attacks with Q of spades
player Player 4 could not defend Q of spades
It is now player Player 1's turn!
Player 3 draws 1 cards. (2 left in deck)
Player 1 attacks with J of diamon